# Практика · Бази даних і SQL

> Лекція: [lecture.html](lecture.html) · Тест: [quiz.html](quiz.html) · ДЗ: [homework.md](homework.md)

Наскрізний приклад той самий, що в лекції, — кавʼярня. Меню з теми 22 переїде
з CSV у базу даних, а поруч зʼявиться друга таблиця — замовлення.

Що зробимо:

1. відкриємо базу в памʼяті (`:memory:`) і базу у файлі — і порівняємо;
2. створимо таблицю `drinks` і заллємо меню через `executemany`;
3. поставимо запити: `where`, `order by`, `limit`;
4. переконаємось, що текст мовчки лягає в стовпець `integer`;
5. зловимо SQL-інʼєкцію на власні очі й побачимо, що параметр `?` її вимикає;
6. порахуємо агрегати й **звіримо `group by` з тим самим підрахунком на словниках**;
7. зшиємо дві таблиці через `join` — і побачимо різницю `inner` / `left`;
8. відкотимо транзакцію після помилки;
9. подивимось `explain query plan` до й після створення індексу;
10. увімкнемо `row_factory` і приберемо за собою.

**Мережа не потрібна:** усі дані ми вигадуємо самі, база живе у тимчасовій теці,
яку наприкінці видаляємо.

## 1 · Тимчасова тека й дві бази

`tempfile.mkdtemp()` створює порожню теку в системному місці для тимчасових файлів.
Так ми нічого не насмітимо в робочій теці — і в кінці приберемо все одним рухом.

Одразу подивимось на дві форми бази. `":memory:"` — база, якої немає на диску: вона
живе в оперативній памʼяті й зникає разом із програмою. Файл на диску — те саме,
але переживає завершення програми.

І ще: версію бібліотеки SQLite беремо з `sqlite3.sqlite_version`, а не з
`sqlite3.version` — останнє застаріло й у Python 3.14 зникне.

In [ ]:
import shutil
import sqlite3
import tempfile
import time
from pathlib import Path

print("бібліотека SQLite:", sqlite3.sqlite_version)

# окрема тека на цей запуск: файли не змішаються з чужими
work_dir = Path(tempfile.mkdtemp(prefix="cafe-sql-"))
db_path = work_dir / "cafe.db"

print("працюємо в:", work_dir)
print("файл бази вже існує:", db_path.exists())

## 2 · База в памʼяті

Найшвидший спосіб щось спробувати. `connect(":memory:")` не створює жодного файлу,
тому таку базу зручно брати для тестів: кожен запуск починається з чистого аркуша.

`execute` виконує один запит, `fetchone()` віддає один рядок результату кортежем.

In [ ]:
memory_con = sqlite3.connect(":memory:")
memory_con.execute("create table sample(name text, price integer)")
memory_con.execute("insert into sample values ('Еспресо', 25)")

row = memory_con.execute("select name, price from sample").fetchone()
print("рядок із памʼяті:", row)
print("тип рядка:", type(row).__name__)

memory_con.close()
print("файлів у теці досі:", list(work_dir.iterdir()))

## 3 · База у файлі й обовʼязковий `commit`

Тепер те саме, але на диску. Тут зʼявляється те, чого не було у файлах із теми 21:
**зміни не потрапляють у базу, доки не викликано `commit()`**.

Спершу навмисно помилимось — зробимо вставку й закриємо зʼєднання без `commit`.
Помилки не буде, а рядка не буде теж.

In [ ]:
con = sqlite3.connect(db_path)
con.execute("""
    create table drinks (
        id          integer primary key,
        name        text    not null,
        price       integer not null,
        category    text    not null,
        description text
    )
""")
con.commit()          # створення таблиці теж треба зафіксувати

# а тепер — вставка БЕЗ commit
con.execute("insert into drinks(name, price, category) values ('Тест', 1, 'кава')")
con.close()

# відкриваємо базу заново й дивимось, чи рядок пережив закриття
con = sqlite3.connect(db_path)
rows_after_close = con.execute("select count(*) from drinks").fetchone()[0]
print("рядків після close() без commit:", rows_after_close)

assert rows_after_close == 0, "незафіксована транзакція мала відкотитись"
print("✅ без commit() зміни зникають — мовчки, без жодної помилки")

## 4 · `with con:` фіксує транзакцію, але **не** закриває зʼєднання

Рефлекс із теми 21 («`with` сам закриє») тут підводить. Перевіримо це прямо:
вийдемо з блоку `with` і спробуємо виконати ще один запит.

In [ ]:
with con:
    con.execute("insert into drinks(name, price, category) values ('Тимчасовий', 1, 'кава')")

# якщо зʼєднання закрилось би — тут був би ProgrammingError
still_alive = con.execute("select count(*) from drinks").fetchone()[0]
print("після виходу з with зʼєднання виконує запити:", still_alive, "рядок у таблиці")

# прибираємо тестовий рядок, щоб меню було чистим
con.execute("delete from drinks")
con.commit()
print("рядків після очищення:", con.execute("select count(*) from drinks").fetchone()[0])
print("✅ with con: — це межа транзакції, а не файлу")

## 5 · Меню кавʼярні через `executemany`

Сім напоїв — той самий список, що в лекції. `executemany` розбирає текст запиту
один раз і виконує всю вставку в одній транзакції.

Стовпець `id` не вказуємо: він оголошений як `integer primary key`, тож база
призначить номери сама.

In [ ]:
menu_rows = [
    ("Еспресо",  25, "кава",  "міцна"),
    ("Капучино", 45, "кава",  "молоко, кориця"),
    ("Латте",    50, "кава",  "багато молока"),
    ("Раф",      55, "кава",  "вершки"),
    ("Чай",      30, "чай",   "травʼяний"),
    ("Матча",    60, "чай",   "японський"),
    ("Какао",    40, "какао", "з молоком"),
]

con.executemany(
    "insert into drinks(name, price, category, description) values (?, ?, ?, ?)",
    menu_rows)
con.commit()

for row in con.execute("select id, name, price, category from drinks order by id"):
    print(row)

print("усього напоїв:", con.execute("select count(*) from drinks").fetchone()[0])

## 6 · `SELECT`: три уточнення до одного запиту

`where` відкидає рядки, `order by` упорядковує, `limit` обрізає. Порядок частин
у тексті запиту суворий — і виконуються вони теж у цьому порядку.

Зверни увагу на останній запит: `limit` спрацював **після** сортування, тому «три
найдорожчі» справді найдорожчі.

In [ ]:
print("уся кава:")
for row in con.execute("select name, price from drinks where category = 'кава'"):
    print("  ", row)

print("\nдорожче за 40 грн, від дешевшого до дорожчого:")
for row in con.execute("select name, price from drinks where price > 40 order by price"):
    print("  ", row)

print("\nтри найдорожчі позиції меню:")
top_three = con.execute(
    "select name, price from drinks order by price desc limit 3").fetchall()
for row in top_three:
    print("  ", row)

assert top_three[0] == ("Матча", 60), "найдорожчою мала бути матча"
print("\n✅ order by + limit дали саме те, що обіцяли")

## 7 · Динамічна типізація: текст у стовпці `integer`

Стовпець `price` оголошено як `integer`. Спробуємо покласти туди три різні речі
й подивимось через функцію `typeof()`, що саме лежить у базі.

Це найважливіша пастка SQLite: рядок із цифр вона перетворить на число сама,
а рядок зі слів залишить текстом — **без помилки й без попередження**.

In [ ]:
probe_con = sqlite3.connect(":memory:")
probe_con.execute("create table probe(price integer)")

for value in (25, "25", "двадцять пʼять"):
    probe_con.execute("insert into probe values (?)", (value,))
    stored, kind = probe_con.execute(
        "select price, typeof(price) from probe order by rowid desc limit 1").fetchone()
    print(f"вставили {value!r:20} → у базі {stored!r:20} typeof={kind}")

# перевіряємо найнеприємніший випадок явно
last_value = probe_con.execute(
    "select price from probe order by rowid desc limit 1").fetchone()[0]
assert isinstance(last_value, str), "текст мав залишитись текстом"
print("\n✅ у числовому стовпці лежить str — арифметика над цінами впаде пізніше")
probe_con.close()

## 8 · SQL-інʼєкція: витік даних, а не видалення таблиці

Робимо маленьку таблицю користувачів і два однакові за наміром запити: один
зібраний f-рядком, другий — із параметром `?`.

Введення `неіснуючий' OR '1'='1` закриває апострофом рядковий літерал, і решта
тексту стає частиною умови. Запит шукав одного користувача — подивимось, скільки
він віддасть.

In [ ]:
users_con = sqlite3.connect(":memory:")
users_con.execute("create table users(login text, role text)")
users_con.executemany("insert into users values (?, ?)", [
    ("olena", "user"),
    ("bohdan", "user"),
    ("admin", "admin"),
])

evil_input = "неіснуючий' OR '1'='1"

# 1. зібрано f-рядком — так робити НЕ можна
unsafe_sql = f"select login, role from users where login='{evil_input}'"
print("запит:", unsafe_sql)
leaked = users_con.execute(unsafe_sql).fetchall()
print("конкатенація повернула рядків:", len(leaked), leaked)

# 2. те саме введення, але як параметр
safe = users_con.execute(
    "select login, role from users where login=?", (evil_input,)).fetchall()
print("параметр повернув рядків:", len(safe), safe)

assert len(leaked) == 3 and ("admin", "admin") in leaked, "інʼєкція мала віддати всіх"
assert safe == [], "параметр мав шукати логін цілком, і не знайти його"
print("\n✅ 3 рядки проти 0: різниця лише в способі підстановки")

## 9 · Класична інʼєкція з коміксів не спрацює

Найвідоміший приклад — `'; drop table users; --`. Спробуємо його й побачимо
справжню поведінку драйвера: `execute` виконує рівно **один** оператор.

Це не означає, що інʼєкцій не буває. Це означає, що небезпечна інʼєкція виглядає
так, як у попередній клітинці: дані крадуть, а не стирають.

In [ ]:
dropper = "x'; drop table users; --"

try:
    users_con.execute(f"select login from users where login='{dropper}'")
    print("оператор виконався — цього не мало статися")
except sqlite3.ProgrammingError as error:
    print("sqlite3.ProgrammingError:", error)

# таблиця на місці
print("рядків у users:", users_con.execute("select count(*) from users").fetchone()[0])

# а от executescript навмисно виконує кілька операторів — і таблиця зникає
users_con.executescript(f"select login from users where login='{dropper}'")
try:
    users_con.execute("select count(*) from users").fetchone()
except sqlite3.OperationalError as error:
    print("після executescript:", error)

users_con.close()
print("\n✅ execute зловив другий оператор, executescript — ні")

## 10 · Агрегація й перевірка `group by` проти чистого Python

Тепер найцінніша клітинка практики. Порахуємо середню ціну по категоріях двома
способами: запитом `group by` і звичайним циклом зі словником — тим самим, який
ми писали в темі 09. Результати мають збігтися до останнього знака.

In [ ]:
# спосіб 1 — база
sql_groups = con.execute("""
    select category, count(*), avg(price)
    from drinks
    group by category
    order by category
""").fetchall()

# спосіб 2 — те саме циклом по рядках, як у темі 09
totals, counts = {}, {}
for name, price, category in con.execute("select name, price, category from drinks"):
    totals[category] = totals.get(category, 0) + price
    counts[category] = counts.get(category, 0) + 1

python_groups = [(category, counts[category], totals[category] / counts[category])
                 for category in sorted(totals)]

print("group by :", sql_groups)
print("цикл     :", python_groups)

assert sql_groups == python_groups, "розрахунок розійшовся!"
print("\n✅ збігається — всередині group by немає магії, лише той самий підрахунок")

## 11 · `having` фільтрує купки, а не рядки

`where` працює **до** групування й нічого не знає про розмір купки.
`having` працює **після** — і тільки в ньому можна згадувати `count(*)`.

У нашому меню одна категорія має єдину позицію: подивимось, як вона зникає.

In [ ]:
all_groups = con.execute(
    "select category, count(*) from drinks group by category order by category").fetchall()
big_groups = con.execute(
    "select category, count(*) from drinks group by category "
    "having count(*) > 1 order by category").fetchall()

print("усі категорії      :", all_groups)
print("де щонайменше дві  :", big_groups)

dropped = [group[0] for group in all_groups if group not in big_groups]
print("відсіяно:", dropped)

assert dropped == ["какао"], "мала випасти саме категорія з однією позицією"
print("\n✅ having прибрав купку з одного рядка")

## 12 · Друга таблиця й `JOIN`

Замовлення живуть окремо від меню й посилаються на нього через `drink_id`.
Дата зберігається текстом у форматі ISO 8601 — власного типу для дати в SQLite
немає, і саме цим займеться тема 24.

Спершу вмикаємо перевірку зовнішніх ключів: за замовчуванням вона **вимкнена**,
і робити це треба на кожному зʼєднанні.

In [ ]:
con.execute("pragma foreign_keys = on")

con.execute("""
    create table orders (
        id         integer primary key,
        drink_id   integer not null references drinks(id),
        qty        integer not null,
        created_at text    not null
    )
""")
con.executemany(
    "insert into orders(drink_id, qty, created_at) values (?, ?, ?)", [
        (1, 2, "2026-03-01T09:12:00"),
        (2, 3, "2026-03-01T10:05:00"),
        (1, 1, "2026-03-01T10:40:00"),
        (4, 1, "2026-03-01T11:15:00"),
    ])
con.commit()

print("замовлень:", con.execute("select count(*) from orders").fetchone()[0])

# перевірка зовнішнього ключа справді працює
try:
    con.execute("insert into orders(drink_id, qty, created_at) values (999, 1, '2026-03-01T12:00:00')")
except sqlite3.IntegrityError as error:
    print("замовлення на неіснуючий напій:", error)
con.rollback()

### `inner join` проти `left join`

`inner join` лишає тільки пари, знайдені з обох боків. `left join` бере всі рядки
лівої таблиці й підставляє `None` там, де пари немає, — саме так шукають те,
чого не замовляли жодного разу.

In [ ]:
sold = con.execute("""
    select drinks.name, orders.qty, drinks.price * orders.qty as total
    from drinks
    join orders on orders.drink_id = drinks.id
    order by orders.id
""").fetchall()

print("inner join — що реально продавалось:")
for row in sold:
    print("  ", row)

never_ordered = con.execute("""
    select drinks.name
    from drinks
    left join orders on orders.drink_id = drinks.id
    where orders.id is null
    order by drinks.name
""").fetchall()

print("\nleft join — чого не замовляли жодного разу:")
for row in never_ordered:
    print("  ", row[0])

assert len(sold) == 4, "мало бути чотири пари напій-замовлення"
assert ("Латте",) in never_ordered, "латте ніхто не замовляв"
print("\n✅ inner віддав 4 пари, left показав напої без жодного замовлення")

## 13 · `UPDATE`, `rowcount` і відкат транзакції

`rowcount` — єдиний спосіб дізнатися, чи запит щось зачепив: `update` з умовою,
якій ніхто не відповідає, помилки не дає.

Далі — головне: підвищимо ціни двома командами, зламаємо процес посередині
й переконаємось, що `rollback()` скасував **обидва** оновлення.

In [ ]:
cursor = con.cursor()

cursor.execute("update drinks set price = ? where name = ?", (52, "Латте"))
print("змінено рядків:", cursor.rowcount)

cursor.execute("update drinks set price = ? where name = ?", (10, "Глінтвейн"))
print("змінено рядків для напою, якого немає:", cursor.rowcount)
con.commit()

price_before = con.execute("select price from drinks where name='Еспресо'").fetchone()[0]

try:
    con.execute("update drinks set price = price + 5 where category = 'кава'")
    con.execute("update drinks set price = price + 5 where category = 'чай'")
    raise ValueError("уявна помилка посеред підвищення цін")
    # рядка con.commit() тут немає навмисно: до нього ми вже не дійшли б
except ValueError as error:
    con.rollback()
    print("зловили:", error, "→ відкочуємо")

price_after = con.execute("select price from drinks where name='Еспресо'").fetchone()[0]
print("ціна еспресо до:", price_before, "· після відкату:", price_after)

assert price_before == price_after, "rollback мав скасувати обидва оновлення"
print("\n✅ транзакція спрацювала за правилом «усе або нічого»")

## 14 · Індекс і план запиту

Сім рядків — замало, щоб побачити різницю, тому зробимо окрему таблицю на
50 000 рядків. `explain query plan` не виконує запит, а показує, **як** база
збирається його виконувати. Текст плану лежить у четвертому полі рядка — `row[3]`.

In [ ]:
big_con = sqlite3.connect(work_dir / "big.db")
big_con.execute("create table items(id integer primary key, name text, price integer)")
big_con.executemany(
    "insert into items(name, price) values (?, ?)",
    [(f"item-{number}", 20 + number % 70) for number in range(50_000)])
big_con.commit()

target = "item-49999"

plan_before = big_con.execute(
    "explain query plan select * from items where name = ?", (target,)).fetchone()
print("план без індексу:", plan_before[3])

big_con.execute("create index idx_items_name on items(name)")
big_con.commit()

plan_after = big_con.execute(
    "explain query plan select * from items where name = ?", (target,)).fetchone()
print("план з індексом :", plan_after[3])

assert plan_before[3].startswith("SCAN"), "без індексу мав бути повний перегляд"
assert "USING INDEX" in plan_after[3], "з індексом мав зʼявитись SEARCH"
print("\n✅ SCAN перетворився на SEARCH — саме це й означає «індекс працює»")

### Скільки це коштує в часі

Час на маленьких таблицях шумить, тому міряємо не один запит, а тисячу — і
дивимось на порядок величини, а не на точне число. У тебе воно буде інше:
воно залежить від машини.

In [ ]:
def search_time(connection, repeats=1000):
    """Середній час одного пошуку за назвою, у мілісекундах."""
    started = time.perf_counter()
    for _ in range(repeats):
        connection.execute("select * from items where name = ?", (target,)).fetchone()
    return (time.perf_counter() - started) / repeats * 1000

with_index = search_time(big_con)
big_con.execute("drop index idx_items_name")
without_index = search_time(big_con, repeats=50)   # без індексу це вже помітно довше

print(f"з індексом  : {with_index:.4f} мс")
print(f"без індексу : {without_index:.4f} мс")
print(f"різниця     : ×{without_index / with_index:.0f}")

assert without_index > with_index, "пошук без індексу не може бути швидшим"
print("\n✅ індекс пришвидшує пошук — і саме тому платить місцем і швидкістю запису")
big_con.close()

## 15 · `row_factory`: рядки як словники

За замовчуванням рядок — кортеж, і значення дістають за номером. Це ламається
від найменшої зміни в списку стовпців. `sqlite3.Row` дає звертання за іменем —
рівно так, як `DictReader` у темі 22.

In [ ]:
con.row_factory = sqlite3.Row
row = con.execute(
    "select name, price, category from drinks where name = 'Капучино'").fetchone()

print("як кортеж :", tuple(row))
print("за іменем :", row["price"])
print("імена стовпців:", row.keys())
print("як словник:", dict(row))

assert row["price"] == row[1], "sqlite3.Row працює і як словник, і як кортеж"
print("\n✅ той самий рядок читається двома способами")

## 16 · Прибираємо за собою

Тимчасові файли не мають переживати запуск зошита. Закриваємо зʼєднання
явно — `with` цього не робить — і видаляємо теку з усім вмістом.

In [ ]:
con.close()

files = sorted(path.name for path in work_dir.iterdir())
print("видаляємо", len(files), "файлів:", files)

shutil.rmtree(work_dir)

print("тека існує:", work_dir.exists())
assert not work_dir.exists(), "тимчасова тека мала зникнути"
print("✅ прибрано")

## Завдання

### 🟢 Рівень 1 — База

Створи в тимчасовій теці базу зі своєю таблицею на 5-7 рядків — **не кавʼярню**.
Наприклад: домашня бібліотека (`title`, `author`, `year`, `genre`). Заллєй дані
через `executemany`, зроби `commit` і надрукуй усі рядки, відсортовані за роком.

**Зроблено, якщо:** зошит виконується без помилок, а `assert` перевіряє, що
`select count(*)` повертає саме стільки рядків, скільки ти вставив.

### 🟡 Рівень 2 — Плюс

До тієї самої таблиці постав три запити: з `where`, з `group by` і з
`having`. Порахуй те саме, що дає `group by`, звичайним циклом зі словником
і звір результати `assert`-ом.

**Зроблено, якщо:** три запити надруковані з результатами, а `assert` про
збіг `group by` із циклом проходить.

### 🔴 Рівень 3 — Виклик

Напиши функцію `find_by(connection, column, value)`, яка шукає рядки за назвою
стовпця, що приходить ззовні. Значення підставляй параметром `?`, а **назву
стовпця** звіряй зі списком дозволених і кидай `ValueError`, якщо її там немає.
Перевір функцію на дозволеному стовпці, на забороненому й на введенні
`неіснуючий' OR '1'='1`.

**Зроблено, якщо:** запит із дозволеним стовпцем повертає рядки, заборонений
стовпець дає `ValueError`, а зловмисне значення повертає **порожній** список,
а не всю таблицю.

### Підказки

* Назву стовпця через `?` підставити не можна — `select * from t where ? = ?`
  не працює. Тому й потрібен список дозволених.
* `explain query plan` можна ставити перед будь-яким `select` — це найшвидший
  спосіб перевірити, чи індекс узагалі використовується.
* Якщо `assert` про кількість рядків падає — перевір, чи є `commit()` після вставки.